# Part 0：数据加载、单月面板与股票池（ver3）

函数定义在 `src/data.py`。ver3 相对 ver2：只转换 11 字段（10 模型 + `amount_21`）、回退门槛配置化（7 基础/3 增强）、训练标签中心化为横截面相对收益。

In [ ]:
import sys
sys.path.insert(0, 'e:/4020prj/code/ver3')
from src.data import *
from src.config import CONFIG


**函数与计算方式（见 `src/data.py`）**

- `build_data(force)`：加载并只转换 `DATA_FIELDS`（11 字段 = 7 基础 + 3 增强 + `amount_21`），构建 `R/universe/industry/rf`，磁盘缓存。返回 `dict`。
- `preprocess_cross_section(X)`：无穷→缺失、`total_mv`取log、`X.quantile([.01,.99])` 向量化截尾、中位数填缺失、有限均值/std 标准化；回退门槛 `base_min_valid=6`、`enh_min_valid=2`。返回 `(Xz, fallback)`。
- `build_monthly_panel(q)`：单个历史月的标准化特征 + `fallback` + 标签 `y`。
- `precompute_panels(months)`：预计算所有历史月面板（磁盘缓存 `panels.pkl`）。
- `assemble_train_panel(train_months, panels)`：由单月面板 concat 出训练面板，并把 `y` 减当月池等权收益（横截面相对收益中心化）。
- `score_from_panel(panel, cand)`：取目标截面特征 + 回退标记（预测用）。

运行：加载数据（首次会读 descriptors 并转换 11 字段）：

In [ ]:
bundle = build_data()
print('R:', bundle['R'].shape, '| D 字段:', len(bundle['D']), '| universe 行:', len(bundle['universe']))
